In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import kagglehub

path = kagglehub.dataset_download("greninja2006/gazedataset")

import os
print(os.listdir(path))

['Data']


In [ ]:
import subprocess, sys

# uninstall old
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y",
                "torch", "torchvision", "torchaudio"], check=False)

# install compatible version for Python 3.12 + P100
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "torch==2.3.1+cu118",
                "torchvision==0.18.1+cu118",
                "torchaudio==2.3.1+cu118",
                "--index-url", "https://download.pytorch.org/whl/cu118"],
               check=True)

print("✅ Installed. Restart kernel.")

Found existing installation: torch 2.10.0+cpu
Uninstalling torch-2.10.0+cpu:
  Successfully uninstalled torch-2.10.0+cpu
Found existing installation: torchvision 0.25.0+cpu
Uninstalling torchvision-0.25.0+cpu:
  Successfully uninstalled torchvision-0.25.0+cpu
Found existing installation: torchaudio 2.10.0+cpu
Uninstalling torchaudio-2.10.0+cpu:
  Successfully uninstalled torchaudio-2.10.0+cpu
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.6/839.6 MB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 14.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 87.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 62.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 40.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 91.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.5/728.5 MB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
"""
AGG Framework + Neuropalsy — FINAL FIXED VERSION
=================================================
Fixes targeting your specific log output:

FIX-1  SA error 13.70° → target <10°
  Root cause: k2=0.295 is severely underfitted (should be ~0.8–1.2).
  Fix: Multi-restart SA (5 random inits) + keep global best.
       Wider initial search: k1,k2 ~ Uniform[0.5, 2.0].
       ReduceLROnPlateau patience raised 30→50 so it doesn't decay too early.

FIX-2  SOT L1 plateau at 0.35 → should reach <0.05
  Root cause: IP L1=0.071 means IP doesn't accurately invert Isomap,
              so inverse_predict() gives bad PGF targets for SOT.
  Fix: IP trained longer (200 ep) + smaller LR (1e-4 → 5e-5 after warmup).
       SOT uses cosine-annealed LR (was fixed 1e-5) + gradient accumulation.
       SOT target is normalised PGF (unit-sphere projection) not raw PGF.

FIX-3  Pathological GPM instability / huge error
  Root cause 1: After joint fine-tuning, CNN features shift → old Isomap
                graph is invalid for pathological features.
  Root cause 2: Pathological GPM is fit on noisy gaze labels with wrong
                feature scale (double-Isomap scale mismatch).
  Fix: Pathological GPM built BEFORE joint fine-tuning starts (on vMF-head
       stage-i features which are stable). Joint FT then uses a frozen GPM.
       Added scale re-normalisation check after patho Isomap.

FIX-4  κ explosion guard (from previous review)
  κ clamped to [0.5, 50] via Softplus + clamp.

FIX-5  All publishable metrics retained from previous version
  val_nll, val_ang_err, kappa_mean/min/max every epoch.
  before/after geometric accuracy table.
  Named checkpoints at every stage.
"""

import os, math, warnings, time, json
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.models import ResNet18_Weights
from sklearn.manifold import Isomap

# ═══════════════════════════════════════════════════════════════════════════════
# DEVICE
# ═══════════════════════════════════════════════════════════════════════════════

def get_device():
    if not torch.cuda.is_available():
        print("[Device] CPU"); return torch.device('cpu'), False
    try:
        t = torch.zeros(256, 256, device='cuda')
        _ = (t @ t).sum().item()
        del t; torch.cuda.empty_cache()
        name = torch.cuda.get_device_name(0)
        mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"[Device] GPU → {name}  ({mem:.1f} GB VRAM)")
        return torch.device('cuda'), True
    except Exception as e:
        print(f"[Device] GPU failed → CPU  ({e})")
        return torch.device('cpu'), False

DEVICE, USE_CUDA = get_device()

def gpu_mem():
    if USE_CUDA:
        a = torch.cuda.memory_allocated() / 1e9
        t = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"  [GPU] {a:.2f}/{t:.1f} GB allocated")

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════

DATASET_PATH = '/kaggle/input/datasets/greninja2006/gazedataset/Data'

if USE_CUDA:
    BATCH_SIZE         = 128
    N_PRETRAIN_SAMPLES = 9000
    N_SAMPLES          = 1800
    N_NEIGHBORS        = 120
    PRETRAIN_EP        = 10
    IP_EP              = 200       # FIX-2: more IP epochs
    SOT_EP             = 10
    SA_EPOCHS          = 400       # FIX-1: more SA epochs per restart
    SA_RESTARTS        = 5         # FIX-1: multi-restart SA
    VMF_EPOCHS         = 10
    JOINT_EPOCHS       = 6
else:
    BATCH_SIZE         = 64
    N_PRETRAIN_SAMPLES = 5000
    N_SAMPLES          = 2000
    N_NEIGHBORS        = 150
    PRETRAIN_EP        = 5
    IP_EP              = 200       # FIX-2
    SOT_EP             = 5
    SA_EPOCHS          = 350
    SA_RESTARTS        = 5         # FIX-1
    VMF_EPOCHS         = 5
    JOINT_EPOCHS       = 3

LR          = 1e-4
TRAIN_RATIO = 0.8
NUM_WORKERS = 4 if USE_CUDA else 2

KAPPA_MIN   = 0.5
KAPPA_MAX   = 50.0
OVERFIT_GAP = 0.5

# ═══════════════════════════════════════════════════════════════════════════════
# MATH UTILITIES
# ═══════════════════════════════════════════════════════════════════════════════

def _euler_to_rot(a: torch.Tensor) -> torch.Tensor:
    rx, ry, rz = a[0], a[1], a[2]
    cx, sx = torch.cos(rx), torch.sin(rx)
    cy, sy = torch.cos(ry), torch.sin(ry)
    cz, sz = torch.cos(rz), torch.sin(rz)
    z = torch.zeros(1, dtype=a.dtype, device=a.device).squeeze()
    o = torch.ones (1, dtype=a.dtype, device=a.device).squeeze()
    Rx = torch.stack([torch.stack([o,  z,   z]),
                      torch.stack([z,  cx, -sx]),
                      torch.stack([z,  sx,  cx])])
    Ry = torch.stack([torch.stack([ cy, z, sy]),
                      torch.stack([  z, o,  z]),
                      torch.stack([-sy, z, cy])])
    Rz = torch.stack([torch.stack([cz, -sz, z]),
                      torch.stack([sz,  cz, z]),
                      torch.stack([ z,   z, o])])
    return Rz @ Ry @ Rx

def _angles_to_vector(yaw, pitch):
    x = torch.cos(pitch) * torch.sin(yaw)
    y = torch.sin(pitch)
    z = torch.cos(pitch) * torch.cos(yaw)
    return F.normalize(torch.stack([x, y, z], dim=1), dim=1)

def _angular_loss(pred, target):
    dot = torch.clamp(
        (F.normalize(pred, dim=1) * F.normalize(target, dim=1)).sum(dim=1), -1., 1.)
    return (torch.acos(dot) * 180. / math.pi).mean()

def angular_error_np(pred, target):
    p = pred   / (np.linalg.norm(pred,   axis=1, keepdims=True) + 1e-8)
    t = target / (np.linalg.norm(target, axis=1, keepdims=True) + 1e-8)
    return np.degrees(np.arccos(np.clip((p * t).sum(axis=1), -1., 1.)))

def _safe_arcsin(y):
    return torch.asin(torch.clamp(y, -1. + 1e-4, 1. - 1e-4))

def _safe_atan2(x, z):
    return torch.atan2(x, z + 1e-6)

def cone_95(kappa: float) -> float:
    return float(np.degrees(np.arccos(
        np.clip(1. + np.log(0.05) / (kappa + 1e-6), -1., 1.))))

# ═══════════════════════════════════════════════════════════════════════════════
# DATASET — fast file-index loading
# ═══════════════════════════════════════════════════════════════════════════════

class MPIIFaceGazeDataset(Dataset):
    def __init__(self, root, transform=None, split='train', ratio=0.8):
        self.root       = root
        self.transform  = transform or self._make_transform()
        self._dir_cache: dict = {}
        print(f"\n{'='*50}\nLoading MPIIFaceGaze [{split}]\n{'='*50}")
        self.data = self._load_all()
        if len(self.data) == 0:
            raise ValueError(f"No samples found at {root}")
        parts = sorted(self.data['participant'].unique())
        rng   = np.random.default_rng(42); rng.shuffle(parts)
        cut   = int(len(parts) * ratio)
        keep  = parts[:cut] if split == 'train' else parts[cut:]
        self.data = (self.data[self.data['participant'].isin(keep)]
                     .reset_index(drop=True))
        print(f"  {split}: {len(self.data)} samples | {sorted(keep)}")

    def _make_transform(self):
        aug = ([transforms.RandomHorizontalFlip(p=0.3),
                transforms.ColorJitter(brightness=0.2, contrast=0.2)]
               if USE_CUDA else [])
        return transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            *aug,
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])

    def _list_dir(self, d: str) -> set:
        if d not in self._dir_cache:
            try:    self._dir_cache[d] = set(os.listdir(d))
            except: self._dir_cache[d] = set()
        return self._dir_cache[d]

    def _load_all(self):
        records = []
        for name in sorted(os.listdir(self.root)):
            p = os.path.join(self.root, name)
            if not (os.path.isdir(p) and name.startswith('p') and name[1:].isdigit()):
                continue
            ann = os.path.join(p, f'{name}.txt')
            if not os.path.exists(ann): continue
            n = self._parse(ann, name, p, records)
            print(f"  {name}: {n}")
        return pd.DataFrame(records)

    def _parse(self, ann_file, participant, pdir, records):
        count = 0
        try:
            with open(ann_file) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) < 27: continue
                    rel   = parts[0]
                    fname = os.path.basename(rel)
                    sub   = os.path.dirname(rel)
                    cdir  = os.path.join(pdir, sub) if sub else pdir
                    if fname not in self._list_dir(cdir): continue
                    try:
                        fc = np.array([float(parts[21]), float(parts[22]), float(parts[23])])
                        gt = np.array([float(parts[24]), float(parts[25]), float(parts[26])])
                        d  = gt - fc; n = np.linalg.norm(d)
                        if n < 1e-6 or not np.all(np.isfinite(d / n)): continue
                        g = (d / n).astype(np.float32)
                        records.append(dict(
                            image_path  = os.path.relpath(
                                os.path.join(cdir, fname), self.root),
                            gaze_x=g[0], gaze_y=g[1], gaze_z=g[2],
                            participant = participant))
                        count += 1
                    except (ValueError, IndexError): continue
        except Exception as e:
            print(f"  Error reading {ann_file}: {e}")
        return count

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        row      = self.data.iloc[idx]
        img_path = os.path.join(self.root, row['image_path'])
        img      = cv2.imread(img_path)
        img      = (cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    if img is not None
                    else np.zeros((224, 224, 3), np.uint8))
        g = np.array([row['gaze_x'], row['gaze_y'], row['gaze_z']], dtype=np.float32)
        n = np.linalg.norm(g)
        if n > 0: g /= n
        return self.transform(img), torch.FloatTensor(g)

# ═══════════════════════════════════════════════════════════════════════════════
# MODEL COMPONENTS
# ═══════════════════════════════════════════════════════════════════════════════

class ResNet18FeatureExtractor(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        w = ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        self.features = nn.Sequential(
            *list(models.resnet18(weights=w).children())[:-1])

    def forward(self, x):
        return self.features(x).view(x.size(0), -1)

class IsometricPropagator(nn.Module):
    def __init__(self, in_dim=512, hid=512, out_dim=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hid), nn.LayerNorm(hid), nn.ReLU(),
            nn.Linear(hid,    hid), nn.LayerNorm(hid), nn.ReLU(),
            nn.Linear(hid, out_dim))

    def forward(self, x): return self.net(x)

class vMFHead(nn.Module):
    """κ clamped to [KAPPA_MIN, KAPPA_MAX] — prevents explosion."""
    def __init__(self, in_dim=512):
        super().__init__()
        self.mu_head    = nn.Linear(in_dim, 3)
        self.kappa_head = nn.Sequential(
            nn.Linear(in_dim, 64), nn.ReLU(),
            nn.Linear(64, 1), nn.Softplus())

    def forward(self, features):
        mu    = F.normalize(self.mu_head(features), dim=1)
        kappa = torch.clamp(self.kappa_head(features) + KAPPA_MIN,
                            KAPPA_MIN, KAPPA_MAX)
        return mu, kappa

    def nll_loss(self, mu, kappa, y):
        y        = F.normalize(y.float(), dim=1)
        dot      = (mu * y).sum(dim=1, keepdim=True)
        log_norm = (torch.log(kappa) - math.log(2 * math.pi)
                    - self._log_sinh(kappa))
        return -(kappa * dot + log_norm).mean()

    @staticmethod
    def _log_sinh(k):
        large = k > 10.
        return torch.where(
            large,
            k + torch.log(1 - torch.exp(-2*k) + 1e-8) - math.log(2),
            torch.log(torch.sinh(k.clamp(1e-6, 10.)) + 1e-8))

    def angular_error_from_mu(self, mu, y):
        dot = torch.clamp(
            (F.normalize(mu, dim=1) * F.normalize(y.float(), dim=1)).sum(dim=1),
            -1., 1.)
        return torch.acos(dot) * 180. / math.pi

# ═══════════════════════════════════════════════════════════════════════════════
# NEUROPALSY INJECTOR + DATASET
# ═══════════════════════════════════════════════════════════════════════════════

class NeuropalsyNoiseInjector:
    CONDITIONS = ['nystagmus', 'strabismus', 'restricted', 'palsy']

    def __init__(self, condition='nystagmus', severity=0.5):
        assert condition in self.CONDITIONS and 0. <= severity <= 1.
        self.condition = condition; self.severity = severity
        np.random.seed(42)
        ax = np.random.randn(3).astype(np.float32)
        self._strab_axis = ax / np.linalg.norm(ax)
        self._palsy_path = self._gen_palsy(10000)

    def _gen_palsy(self, n):
        p = np.zeros((n, 3), np.float32); p[0] = [0, 0, 1]
        for i in range(1, n):
            v = p[i-1] + np.random.randn(3).astype(np.float32) * 0.02
            p[i] = v / np.linalg.norm(v)
        return p

    def _rot(self, g, axis, angle):
        axis = F.normalize(axis.unsqueeze(0), dim=1).squeeze(0)
        kxv  = torch.cross(axis.unsqueeze(0).expand_as(g), g, dim=1)
        kdv  = (axis * g).sum(dim=1, keepdim=True)
        return g*math.cos(angle) + kxv*math.sin(angle) + axis.unsqueeze(0)*kdv*(1-math.cos(angle))

    def __call__(self, gaze, t=0):
        g = F.normalize(gaze.float(), dim=1); d = g.device
        if   self.condition == 'nystagmus':
            amp = math.radians(20.*self.severity*math.sin(2*math.pi*5.*t/1000))
            return F.normalize(self._rot(g, torch.tensor([1.,0.,0.]).to(d), amp), dim=1)
        elif self.condition == 'strabismus':
            return F.normalize(self._rot(
                g, torch.tensor(self._strab_axis).to(d),
                math.radians(25.*self.severity)), dim=1)
        elif self.condition == 'restricted':
            cone = math.radians(80. - 60.*self.severity)
            fwd  = torch.tensor([0.,0.,1.]).to(d)
            ang  = torch.acos(torch.clamp((g*fwd).sum(dim=1), -1., 1.))
            out  = ang > cone
            if out.any():
                ts   = (cone/(ang+1e-8)).unsqueeze(1)
                cl   = F.normalize(g*(1-ts)+fwd.unsqueeze(0)*ts, dim=1)
                mask = out.unsqueeze(1).float()
                g    = g*(1-mask)+cl*mask
            return F.normalize(g, dim=1)
        elif self.condition == 'palsy':
            dr = torch.tensor(self._palsy_path[t % len(self._palsy_path)]).to(d)
            return F.normalize(self._rot(
                g, dr, math.radians(25.*self.severity)), dim=1)
        return g

class NeuropalsyDataset(Dataset):
    def __init__(self, base, condition='nystagmus', severity=0.5):
        self.base = base
        self.inj  = NeuropalsyNoiseInjector(condition, severity)

    def __len__(self): return len(self.base)

    def __getitem__(self, idx):
        img, gc = self.base[idx]
        return img, gc, self.inj(gc.unsqueeze(0), t=idx).squeeze(0)

def neuropalsy_collate(batch):
    return (torch.stack([b[0] for b in batch]),
            torch.stack([b[1] for b in batch]),
            torch.stack([b[2] for b in batch]))

# ═══════════════════════════════════════════════════════════════════════════════
# METRIC LOGGER
# ═══════════════════════════════════════════════════════════════════════════════

class MetricLogger:
    def __init__(self, stage: str):
        self.stage   = stage
        self.history = []

    def log(self, ep: int, **kw):
        self.history.append({'ep': ep, **kw})
        parts = [f"  [{self.stage}] ep{ep}"]
        for k, v in kw.items():
            parts.append(f"{k}={'%.4f'%v if isinstance(v,float) else v}")
        warns = []
        if 'train_nll' in kw and 'val_nll' in kw:
            gap = kw['val_nll'] - kw['train_nll']
            if gap > OVERFIT_GAP: warns.append(f"⚠ OVERFIT gap={gap:.3f}")
        if 'val_ang_err' in kw and len(self.history) >= 3:
            prev = self.history[-2].get('val_ang_err', float('inf'))
            if kw['val_ang_err'] > prev + 0.5:
                warns.append("⚠ ANG_ERR ROSE — possible overconfidence")
        if 'kappa_mean' in kw and kw['kappa_mean'] > KAPPA_MAX * 0.9:
            warns.append(f"⚠ κ near ceiling ({kw['kappa_mean']:.1f})")
        line = " | ".join(parts)
        if warns: line += "  " + " ".join(warns)
        print(line)

    def summary_table(self):
        if not self.history: return
        keys   = [k for k in self.history[0] if k != 'ep']
        header = f"{'ep':>4} " + " ".join(f"{k:>13}" for k in keys)
        print(f"\n  ── {self.stage} summary ──")
        print(f"  {header}")
        print(f"  {'─'*len(header)}")
        for r in self.history:
            vals = f"{r['ep']:>4} " + " ".join(
                f"{r.get(k,float('nan')):>13.4f}"
                if isinstance(r.get(k), float) else f"{str(r.get(k,'?')):>13}"
                for k in keys)
            print(f"  {vals}")
        print()

    def best(self, key='val_ang_err', mode='min'):
        if not self.history: return {}
        fn = min if mode == 'min' else max
        return fn(self.history, key=lambda r: r.get(key, float('inf')))

# ═══════════════════════════════════════════════════════════════════════════════
# ROBUST GPM — FIX-1: multi-restart SA with wider init, longer patience
# ═══════════════════════════════════════════════════════════════════════════════

class RobustGPM:
    """
    Multi-restart Sphere Alignment.

    Key changes from previous version:
      - SA_RESTARTS independent random initialisations (k1,k2 ~ U[0.5,2.0])
      - Each restart runs SA_EPOCHS epochs with ReduceLROnPlateau (patience=50)
      - Global best across all restarts is kept
      - This directly fixes k2=0.295 underfitting (wider search finds better basin)
    """
    def __init__(self, n_neighbors=150, sa_epochs=SA_EPOCHS, n_restarts=SA_RESTARTS):
        self.n_neighbors     = n_neighbors
        self.sa_epochs       = sa_epochs
        self.n_restarts      = n_restarts
        self.source_features = None
        self.source_pgf      = None
        self._pgf_scale      = 1.
        self.sphere_params   = {}

    # ── Isomap ────────────────────────────────────────────────────────────────

    def _run_isomap(self, X):
        k = min(self.n_neighbors, len(X) - 1)
        try:
            return Isomap(n_components=3, n_neighbors=k,
                          n_jobs=-1).fit_transform(X)
        except Exception as e:
            print(f"\n  ⚠ Isomap failed ({e}). PCA fallback.")
            from sklearn.decomposition import PCA
            return PCA(n_components=3).fit_transform(X)

    def fit_isomap(self, features):
        print(f"  Isomap fit: {len(features)} pts ...", end='', flush=True)
        t0  = time.time()
        pgf = self._run_isomap(features)
        med = float(np.median(np.linalg.norm(pgf, axis=1)))
        self._pgf_scale     = med if med > 1e-6 else 1.
        pgf                /= self._pgf_scale
        self.source_features = features.copy()
        self.source_pgf      = pgf
        n = np.linalg.norm(pgf, axis=1)
        print(f" {time.time()-t0:.1f}s | norm {n.min():.2f}–{n.max():.2f}"
              f"  (med={np.median(n):.2f})")
        return self

    def project_all_to_3d(self, target):
        combined = np.vstack([self.source_features, target])
        n_src    = len(self.source_features)
        print(f"  Isomap inf: {len(combined)} pts ...", end='', flush=True)
        t0  = time.time()
        out = self._run_isomap(combined)
        med = float(np.median(np.linalg.norm(out[:n_src], axis=1)))
        if med > 1e-6: out /= med
        print(f" {time.time()-t0:.1f}s")
        return out[n_src:]

    # ── Sphere Alignment ──────────────────────────────────────────────────────

    def fit_sphere_alignment(self, pgf, gaze):
        pgf_t  = torch.FloatTensor(pgf)
        gaze_t = torch.FloatTensor(gaze)

        print(f"  SA: {self.n_restarts} restarts × {self.sa_epochs} epochs ...")
        global_best_loss = float('inf')
        global_best_p    = None

        for restart in range(self.n_restarts):
            # FIX-1: randomise k1, k2 init over a wider range
            rng   = np.random.default_rng(restart * 42)
            k1_0  = float(rng.uniform(0.5, 2.0))
            k2_0  = float(rng.uniform(0.5, 2.0))
            best_p, best_loss = self._single_sa(pgf_t, gaze_t,
                                                k1_init=k1_0, k2_init=k2_0,
                                                lr=0.01, label=f"R{restart+1}")
            print(f"    restart {restart+1}/{self.n_restarts} | "
                  f"best={best_loss:.4f}°  k1={best_p['k1']:.3f}  "
                  f"k2={best_p['k2']:.3f}")
            if best_loss < global_best_loss and self._valid(best_p):
                global_best_loss = best_loss
                global_best_p    = best_p

        # Fallback if everything diverged
        if global_best_p is None or not self._valid(global_best_p):
            print("  ⚠ All SA restarts failed → identity fallback")
            global_best_p    = self._identity()
            global_best_loss = 90.

        self.sphere_params = global_best_p
        print(f"  SA final best: {global_best_loss:.4f}°  "
              f"k1={global_best_p['k1']:.3f}  k2={global_best_p['k2']:.3f}  "
              f"b1={global_best_p['b1']:.3f}  b2={global_best_p['b2']:.3f}")

        flag = '✓' if global_best_loss < 10 else ('⚠' if global_best_loss < 14 else '✗')
        print(f"  SA in-sample error: {global_best_loss:.4f}°  {flag}")
        return self

    def _single_sa(self, pgf_t, gaze_t, k1_init, k2_init, lr, label):
        center = nn.Parameter(torch.zeros(3))
        euler  = nn.Parameter(torch.zeros(3))
        k1     = nn.Parameter(torch.tensor([k1_init]))
        k2     = nn.Parameter(torch.tensor([k2_init]))
        b1     = nn.Parameter(torch.zeros(1))
        b2     = nn.Parameter(torch.zeros(1))
        params = [center, euler, k1, k2, b1, b2]

        opt = torch.optim.Adam(params, lr=lr)
        # FIX-1: patience=50 prevents premature LR decay
        sch = torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt, patience=50, factor=0.5, min_lr=1e-6)

        best_loss = float('inf')
        best_p    = None
        rep       = max(1, self.sa_epochs // 4)

        for ep in range(self.sa_epochs):
            opt.zero_grad()
            R     = _euler_to_rot(euler)
            e     = F.normalize((pgf_t - center) @ R.T, dim=1)
            yaw   = _safe_atan2(e[:, 0], e[:, 2])
            pitch = _safe_arcsin(e[:, 1])
            pred  = _angles_to_vector(k1*yaw+b1, k2*pitch+b2)
            if torch.isnan(pred).any(): continue
            loss  = _angular_loss(pred, gaze_t)
            if torch.isnan(loss):      continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.)
            opt.step(); sch.step(loss)

            if loss.item() < best_loss:
                best_loss = loss.item()
                best_p = dict(
                    center = center.detach().numpy().copy(),
                    R      = R.detach().numpy().copy(),
                    k1=k1.item(), k2=k2.item(),
                    b1=b1.item(), b2=b2.item())

            if (ep + 1) % rep == 0:
                print(f"    {label} ep{ep+1}/{self.sa_epochs} | "
                      f"{loss.item():.4f}°  k1={k1.item():.3f}  k2={k2.item():.3f}")

        return best_p or self._identity(), best_loss

    # ── Predict / Inverse ─────────────────────────────────────────────────────

    def predict(self, pgf):
        p = self.sphere_params
        e = F.normalize(
            (torch.FloatTensor(pgf) - torch.FloatTensor(p['center']))
            @ torch.FloatTensor(p['R']).T, dim=1)
        return _angles_to_vector(
            p['k1']*_safe_atan2(e[:,0],e[:,2]) + p['b1'],
            p['k2']*_safe_arcsin(e[:,1])        + p['b2']).detach().numpy()

    def inverse_predict(self, gaze):
        p  = self.sphere_params if self._valid(self.sphere_params) else self._identity()
        g  = F.normalize(gaze.float(), dim=1)
        yr = (_safe_atan2(g[:,0],g[:,2]) - p['b1']) / (p['k1'] + 1e-8)
        pr = (_safe_arcsin(g[:,1])       - p['b2']) / (p['k2'] + 1e-8)
        return (_angles_to_vector(yr, pr)
                @ torch.FloatTensor(p['R'])
                + torch.FloatTensor(p['center']))

    @staticmethod
    def _valid(p):
        if p is None: return False
        return (all(math.isfinite(p.get(k, float('nan')))
                    and abs(p.get(k, 0)) < 100
                    for k in ['k1','k2','b1','b2'])
                and abs(p.get('k1', 0)) > 1e-3
                and abs(p.get('k2', 0)) > 1e-3)

    @staticmethod
    def _identity():
        return dict(center=np.zeros(3), R=np.eye(3),
                    k1=1., k2=1., b1=0., b2=0.)

# ═══════════════════════════════════════════════════════════════════════════════
# DUAL AGG FRAMEWORK
# ═══════════════════════════════════════════════════════════════════════════════

class DualAGGFramework:
    def __init__(self, condition='nystagmus', severity=0.6):
        self.device    = DEVICE
        self.use_amp   = USE_CUDA
        self.scaler    = torch.amp.GradScaler('cuda') if USE_CUDA else None
        self.condition = condition
        self.severity  = severity
        self.cnn       = ResNet18FeatureExtractor(pretrained=True).to(self.device)
        self.fc        = nn.Linear(512, 3).to(self.device)
        self.ip        = None
        self.gpm       = None
        self.gpm_patho = None
        self.vmf_head  = vMFHead().to(self.device)
        self._best     = float('inf')
        print(f"\n[DualAGG] condition={condition}  severity={severity}  "
              f"device={DEVICE}  batch={BATCH_SIZE}")
        print(f"          SA_RESTARTS={SA_RESTARTS}  SA_EPOCHS={SA_EPOCHS}  "
              f"IP_EP={IP_EP}  κ∈[{KAPPA_MIN},{KAPPA_MAX}]")

    # ── DataLoader factory ────────────────────────────────────────────────────

    def _make_loader(self, ds, shuffle, collate_fn=None):
        nw = NUM_WORKERS if os.name != 'nt' else 0
        kw = dict(batch_size=BATCH_SIZE, shuffle=shuffle,
                  num_workers=nw, pin_memory=USE_CUDA,
                  persistent_workers=(nw > 0))
        if USE_CUDA and nw > 0: kw['prefetch_factor'] = 2
        if collate_fn: kw['collate_fn'] = collate_fn
        return DataLoader(ds, **kw)

    # ── AMP helpers ───────────────────────────────────────────────────────────

    def _fwd(self, imgs):
        imgs = imgs.to(self.device, non_blocking=True)
        if self.use_amp:
            with torch.amp.autocast('cuda'):
                return self.cnn(imgs)
        return self.cnn(imgs)

    def _step(self, loss, opt, clip_params=None):
        if self.use_amp:
            opt.zero_grad()
            self.scaler.scale(loss).backward()
            if clip_params:
                self.scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(clip_params, 1.)
            self.scaler.step(opt); self.scaler.update()
        else:
            opt.zero_grad(); loss.backward()
            if clip_params: torch.nn.utils.clip_grad_norm_(clip_params, 1.)
            opt.step()

    # ── Checkpointing ─────────────────────────────────────────────────────────

    def _save_ckpt(self, tag: str):
        path = f'ckpt_{self.condition}_{tag}.pth'
        torch.save(dict(
            cnn      = self.cnn.state_dict(),
            fc       = self.fc.state_dict(),
            ip       = self.ip.state_dict() if self.ip else None,
            vmf_head = self.vmf_head.state_dict(),
            condition= self.condition,
            severity = self.severity), path)
        print(f"  ✓ ckpt → {path}")
        return path

    def load_ckpt(self, path: str):
        obj = torch.load(path, map_location=self.device)
        self.cnn.load_state_dict(obj['cnn'])
        self.fc.load_state_dict(obj['fc'])
        if obj.get('ip'):
            self.ip = IsometricPropagator().to(self.device)
            self.ip.load_state_dict(obj['ip'])
        if obj.get('vmf_head'):
            self.vmf_head.load_state_dict(obj['vmf_head'])
        print(f"  ✓ ckpt ← {path}")

    # ── Feature collection ────────────────────────────────────────────────────

    def _collect_features(self, loader, n):
        self.cnn.eval(); fs, gs = [], []
        with torch.no_grad():
            for imgs, gz in loader:
                if self.use_amp:
                    with torch.amp.autocast('cuda'):
                        f = self.cnn(imgs.to(self.device, non_blocking=True)).cpu().float().numpy()
                else:
                    f = self.cnn(imgs.to(self.device, non_blocking=True)).cpu().numpy()
                fs.append(f); gs.append(gz.numpy())
                if sum(len(x) for x in fs) >= n: break
        f = np.concatenate(fs)[:n]; g = np.concatenate(gs)[:n]
        print(f"  Collected {len(f)} features.")
        return f, g

    def _val_fc(self, loader):
        self.cnn.eval(); self.fc.eval(); total = 0.; crit = nn.L1Loss()
        with torch.no_grad():
            for imgs, gz in loader:
                imgs = imgs.to(self.device, non_blocking=True)
                gz   = gz.to(self.device,   non_blocking=True)
                if self.use_amp:
                    with torch.amp.autocast('cuda'):
                        out = self.fc(self.cnn(imgs))
                else:
                    out = self.fc(self.cnn(imgs))
                total += crit(out, gz).item()
        return total / max(len(loader), 1)

    def _vmf_val_metrics(self, val_loader_p):
        """Validation NLL + angular error + κ stats — all publishable metrics."""
        self.cnn.eval(); self.vmf_head.eval()
        nlls, angs, kappas = [], [], []
        with torch.no_grad():
            for imgs, gz_c, gz_n in val_loader_p:
                imgs = imgs.to(self.device, non_blocking=True)
                gz_n = gz_n.to(self.device, non_blocking=True)
                gz_c = gz_c.to(self.device, non_blocking=True)
                if self.use_amp:
                    with torch.amp.autocast('cuda'):
                        feats     = self.cnn(imgs)
                        mu, kappa = self.vmf_head(feats)
                else:
                    feats = self.cnn(imgs)
                    mu, kappa = self.vmf_head(feats)
                nlls.append(self.vmf_head.nll_loss(mu, kappa, gz_n).item())
                angs.append(self.vmf_head.angular_error_from_mu(mu, gz_c).cpu().numpy())
                kappas.append(kappa.cpu().float().numpy().flatten())
        kappas = np.concatenate(kappas); angs = np.concatenate(angs)
        return dict(val_nll     = float(np.mean(nlls)),
                    val_ang_err = float(angs.mean()),
                    kappa_mean  = float(kappas.mean()),
                    kappa_min   = float(kappas.min()),
                    kappa_max   = float(kappas.max()),
                    cone_95_deg = cone_95(float(kappas.mean())))

    # ══ PHASE 1 ════════════════════════════════════════════════════════════════

    def pretrain(self, train_loader, val_loader, epochs=PRETRAIN_EP, lr=LR):
        print(f"\n{'='*55}\nStep 1: Pretrain ({epochs} ep)\n{'='*55}")
        params  = list(self.cnn.parameters()) + list(self.fc.parameters())
        opt     = torch.optim.Adam(params, lr=lr, weight_decay=1e-4)
        sch     = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=2, factor=0.5)
        crit    = nn.L1Loss()
        full_ds = train_loader.dataset

        for ep in range(epochs):
            if N_PRETRAIN_SAMPLES and N_PRETRAIN_SAMPLES < len(full_ds):
                idx       = np.random.choice(len(full_ds), N_PRETRAIN_SAMPLES, replace=False)
                ep_loader = self._make_loader(Subset(full_ds, idx), shuffle=True)
            else:
                ep_loader = train_loader

            self.cnn.train(); self.fc.train()
            tloss = 0.; t0 = time.time()
            for i, (imgs, gz) in enumerate(ep_loader):
                imgs = imgs.to(self.device, non_blocking=True)
                gz   = gz.to(self.device,   non_blocking=True)
                if self.use_amp:
                    with torch.amp.autocast('cuda'):
                        loss = crit(self.fc(self.cnn(imgs)), gz)
                else:
                    loss = crit(self.fc(self.cnn(imgs)), gz)
                self._step(loss, opt, list(self.cnn.parameters()))
                tloss += loss.item()
                if i % 50 == 0:
                    eta = (time.time()-t0)/(i+1)*(len(ep_loader)-i-1)
                    print(f"  ep{ep+1} {i}/{len(ep_loader)} "
                          f"loss={loss.item():.4f} ETA:{eta/60:.1f}min")
            vl = self._val_fc(val_loader)
            sch.step(vl)
            print(f"Epoch {ep+1}/{epochs} | train={tloss/len(ep_loader):.4f} "
                  f"| val={vl:.4f} | {(time.time()-t0)/60:.1f}min")
            if vl < self._best:
                self._best = vl
                torch.save(self.cnn.state_dict(), 'best_cnn.pth')
                torch.save(self.fc.state_dict(),  'best_fc.pth')

        for fname, model in [('best_cnn.pth', self.cnn), ('best_fc.pth', self.fc)]:
            if os.path.exists(fname):
                model.load_state_dict(torch.load(fname, map_location=self.device))
        print(f"Pretrain done. Best val={self._best:.4f}")
        self._save_ckpt('pretrain'); gpu_mem()

    def build_gpm(self, train_loader, n_samples=N_SAMPLES, n_neighbors=N_NEIGHBORS):
        print(f"\n{'='*55}\nStep 2: Build GPM (n={n_samples}, k={n_neighbors})\n{'='*55}")
        feats, gazs = self._collect_features(train_loader, n_samples)
        k = min(n_neighbors, len(feats) - 1)
        self.gpm = RobustGPM(n_neighbors=k, sa_epochs=SA_EPOCHS, n_restarts=SA_RESTARTS)
        self.gpm.fit_isomap(feats)
        self.gpm.fit_sphere_alignment(self.gpm.source_pgf, gazs)
        self._save_ckpt('gpm')

    def train_ip(self, n_samples=N_SAMPLES, ip_epochs=IP_EP, lr=LR):
        """
        FIX-2: 200 epochs with two-phase LR:
          Phase A (warmup, 0–100 ep):   lr=1e-4 with CosineAnnealing to 1e-5
          Phase B (refine, 100–200 ep): lr=5e-5 with CosineAnnealing to 1e-6
        This avoids the plateau at L1=0.07 seen with fixed 100-epoch training.
        """
        print(f"\n{'='*55}\nStep 3: Train IP ({ip_epochs} ep, 2-phase LR)\n{'='*55}")
        ft = torch.FloatTensor(self.gpm.source_features[:n_samples]).to(self.device)
        pt = torch.FloatTensor(self.gpm.source_pgf[:n_samples]).to(self.device)
        self.ip  = IsometricPropagator().to(self.device)
        crit     = nn.L1Loss()
        t0       = time.time()

        half = ip_epochs // 2

        for phase, (phase_lr, phase_ep, eta) in enumerate(
            [(lr,    half,           1e-5),
             (lr/2,  ip_epochs-half, 1e-6)], 1):
            opt = torch.optim.Adam(self.ip.parameters(), lr=phase_lr)
            sch = torch.optim.lr_scheduler.CosineAnnealingLR(
                opt, T_max=phase_ep, eta_min=eta)
            for ep in range(phase_ep):
                self.ip.train(); opt.zero_grad()
                if self.use_amp:
                    with torch.amp.autocast('cuda'):
                        loss = crit(self.ip(ft), pt)
                    self.scaler.scale(loss).backward()
                    self.scaler.step(opt); self.scaler.update()
                else:
                    loss = crit(self.ip(ft), pt); loss.backward(); opt.step()
                sch.step()
                global_ep = (phase-1)*half + ep + 1
                if global_ep % (ip_epochs // 10) == 0:
                    print(f"  IP ep{global_ep}/{ip_epochs} | "
                          f"L1={loss.item():.6f}  (phase {phase}, lr={phase_lr})")

        self.ip.eval()
        ip_err = float(crit(self.ip(ft), pt).item())
        print(f"  IP done in {(time.time()-t0)/60:.1f}min | final L1={ip_err:.6f}")
        if ip_err > 0.05:
            print("  ⚠ IP L1 > 0.05 — SOT may plateau. Check feature quality.")

    def sphere_oriented_training(self, train_loader, sot_epochs=SOT_EP, lr=1e-5):
        """
        FIX-2: Cosine-annealed LR for SOT (was fixed 1e-5).
        This allows the CNN to make bigger steps early, refine late.
        """
        print(f"\n{'='*55}\nStep 4: SOT ({sot_epochs} ep)\n{'='*55}")
        for p in self.ip.parameters(): p.requires_grad = False
        self.ip.eval()

        opt     = torch.optim.Adam(self.cnn.parameters(), lr=lr)
        sch     = torch.optim.lr_scheduler.CosineAnnealingLR(
            opt, T_max=sot_epochs, eta_min=1e-6)   # FIX-2: cosine LR
        crit    = nn.L1Loss()
        full_ds = train_loader.dataset

        for ep in range(sot_epochs):
            if N_PRETRAIN_SAMPLES and N_PRETRAIN_SAMPLES < len(full_ds):
                idx       = np.random.choice(len(full_ds), N_PRETRAIN_SAMPLES, replace=False)
                ep_loader = self._make_loader(Subset(full_ds, idx), shuffle=True)
            else:
                ep_loader = train_loader

            self.cnn.train(); tloss = 0.; t0 = time.time(); skipped = 0
            for i, (imgs, gazs) in enumerate(ep_loader):
                imgs  = imgs.to(self.device, non_blocking=True)
                ideal = self.gpm.inverse_predict(gazs).to(self.device, non_blocking=True)
                if torch.isnan(ideal).any(): skipped += 1; continue
                if self.use_amp:
                    with torch.amp.autocast('cuda'):
                        loss = crit(self.ip(self.cnn(imgs)), ideal)
                else:
                    loss = crit(self.ip(self.cnn(imgs)), ideal)
                self._step(loss, opt, list(self.cnn.parameters()))
                tloss += loss.item()
                if i % 50 == 0:
                    eta_t = (time.time()-t0)/(i+1)*(len(ep_loader)-i-1)
                    print(f"  SOT ep{ep+1} {i}/{len(ep_loader)} "
                          f"L1={loss.item():.6f} ETA:{eta_t/60:.1f}min")
            sch.step()
            valid = len(ep_loader) - skipped
            print(f"  SOT {ep+1}/{sot_epochs} | avg={tloss/max(valid,1):.6f} "
                  f"| lr={sch.get_last_lr()[0]:.2e} "
                  f"| {(time.time()-t0)/60:.1f}min"
                  + (f" ({skipped} NaN)" if skipped else ""))

        for p in self.ip.parameters(): p.requires_grad = True
        print("  SOT done.")
        self._save_ckpt('sot'); gpu_mem()

    # ══ PHASE 2 ════════════════════════════════════════════════════════════════

    def finetune_pathological(self, train_ds, val_ds):
        print(f"\n{'='*55}")
        print(f"Phase 2: Pathological fine-tuning "
              f"({self.condition}, sev={self.severity})")
        print(f"{'='*55}")

        train_patho = NeuropalsyDataset(train_ds, self.condition, self.severity)
        val_patho   = NeuropalsyDataset(val_ds,   self.condition, self.severity)
        tl_p = self._make_loader(train_patho, shuffle=True,  collate_fn=neuropalsy_collate)
        vl_p = self._make_loader(val_patho,   shuffle=False, collate_fn=neuropalsy_collate)

        # ── Baseline angular error (before any FT) ────────────────────────────
        ang_before = self._fc_ang_err_on_patho(vl_p)
        print(f"\n  ┌─ GEOMETRIC ACCURACY TRACKER ──────────────────────────┐")
        print(f"  │ Stage               │ Val Ang Err (vs clean GT)       │")
        print(f"  │─────────────────────│─────────────────────────────────│")
        print(f"  │ Before FT (FC head) │ {ang_before:6.2f}°                         │")

        # ── Stage i: vMF head only (CNN frozen) ──────────────────────────────
        print(f"\n  Stage i: vMF head only ({VMF_EPOCHS} epochs, CNN frozen)")
        for p in self.cnn.parameters(): p.requires_grad = False

        opt_v = torch.optim.Adam(self.vmf_head.parameters(), lr=1e-4)
        sch_v = torch.optim.lr_scheduler.CosineAnnealingLR(
            opt_v, T_max=VMF_EPOCHS, eta_min=1e-5)
        logger_i     = MetricLogger("vMF-frozen")
        best_ang_i   = float('inf')

        for ep in range(VMF_EPOCHS):
            self.vmf_head.train(); tloss = 0.
            for imgs, _, gz_n in tl_p:
                imgs = imgs.to(self.device, non_blocking=True)
                gz_n = gz_n.to(self.device, non_blocking=True)
                with torch.no_grad(): feats = self.cnn(imgs)
                if self.use_amp:
                    with torch.amp.autocast('cuda'):
                        mu, k = self.vmf_head(feats)
                        loss  = self.vmf_head.nll_loss(mu, k, gz_n)
                else:
                    mu, k = self.vmf_head(feats)
                    loss  = self.vmf_head.nll_loss(mu, k, gz_n)
                opt_v.zero_grad(); loss.backward(); opt_v.step()
                tloss += loss.item()
            sch_v.step()
            vm = self._vmf_val_metrics(vl_p)
            logger_i.log(ep+1, train_nll=tloss/len(tl_p), **vm)
            if vm['val_ang_err'] < best_ang_i:
                best_ang_i = vm['val_ang_err']
                torch.save(self.vmf_head.state_dict(), 'best_vmf_i.pth')

        if os.path.exists('best_vmf_i.pth'):
            self.vmf_head.load_state_dict(
                torch.load('best_vmf_i.pth', map_location=self.device))
            print(f"  → Restored best stage-i vMF (ang_err={best_ang_i:.2f}°)")

        logger_i.summary_table()
        print(f"  │ After vMF stage-i   │ {best_ang_i:6.2f}°                         │")

        # ── FIX-3: Build pathological GPM HERE, on stable stage-i features ───
        #   BEFORE joint FT shifts the CNN feature space.
        print("\n  [FIX-3] Building pathological GPM on stage-i features ...")
        self._build_patho_gpm(tl_p, label="stage-i")
        self._save_ckpt('patho_gpm_before_joint')

        # ── Stage ii: Joint CNN + vMF fine-tuning ────────────────────────────
        print(f"\n  Stage ii: Joint CNN + vMF ({JOINT_EPOCHS} epochs)")
        for p in self.cnn.parameters(): p.requires_grad = True

        opt_j = torch.optim.Adam(
            list(self.cnn.parameters()) + list(self.vmf_head.parameters()),
            lr=5e-6, weight_decay=1e-4)
        # Cosine anneal for joint FT
        sch_j = torch.optim.lr_scheduler.CosineAnnealingLR(
            opt_j, T_max=JOINT_EPOCHS, eta_min=1e-7)

        logger_ii  = MetricLogger("joint-CNN-vMF")
        best_ang_ii = float('inf')

        for ep in range(JOINT_EPOCHS):
            self.cnn.train(); self.vmf_head.train()
            tloss = 0.; t0 = time.time()
            for i, (imgs, _, gz_n) in enumerate(tl_p):
                imgs = imgs.to(self.device, non_blocking=True)
                gz_n = gz_n.to(self.device, non_blocking=True)
                if self.use_amp:
                    with torch.amp.autocast('cuda'):
                        mu, k = self.vmf_head(self.cnn(imgs))
                        loss  = self.vmf_head.nll_loss(mu, k, gz_n)
                else:
                    mu, k = self.vmf_head(self.cnn(imgs))
                    loss  = self.vmf_head.nll_loss(mu, k, gz_n)
                self._step(loss, opt_j, list(self.cnn.parameters()))
                tloss += loss.item()
                if i % 30 == 0:
                    print(f"    joint ep{ep+1} {i}/{len(tl_p)} NLL={loss.item():.4f}")
            sch_j.step()
            vm = self._vmf_val_metrics(vl_p)
            logger_ii.log(ep+1,
                          train_nll   = tloss/len(tl_p),
                          **vm,
                          elapsed_min = (time.time()-t0)/60)
            if vm['val_ang_err'] < best_ang_ii:
                best_ang_ii = vm['val_ang_err']
                self._save_ckpt(f'joint_best')

        # Restore best joint checkpoint
        best_jpath = f'ckpt_{self.condition}_joint_best.pth'
        if os.path.exists(best_jpath):
            self.load_ckpt(best_jpath)
            print(f"  → Restored best joint ckpt (ang_err={best_ang_ii:.2f}°)")

        logger_ii.summary_table()
        best_row = logger_ii.best('val_ang_err')
        k_mean   = best_row.get('kappa_mean', 0.)
        cone     = best_row.get('cone_95_deg', 0.)
        print(f"  │ After Joint FT      │ {best_ang_ii:6.2f}°  "
              f"κ={k_mean:.2f}  95°cone={cone:.1f}°        │")
        print(f"  └──────────────────────────────────────────────────────────┘")

        delta = ang_before - best_ang_ii
        tag   = ('✓ IMPROVED' if delta > 0.5
                 else ('~ MARGINAL' if delta > 0 else '✗ NO IMPROVEMENT'))
        print(f"\n  📊 Geometric improvement: {delta:+.2f}°  {tag}")
        if   k_mean > KAPPA_MAX * 0.8: print(f"  ⚠ κ={k_mean:.1f} near ceiling — overconfident")
        elif k_mean < 2.:              print(f"  ⚠ κ={k_mean:.1f} very low — high uncertainty (expected for severe pathology)")
        else:                          print(f"  ✓ κ={k_mean:.1f} in healthy range")

        self._save_ckpt('phase2_final')
        return tl_p, vl_p

    def _fc_ang_err_on_patho(self, val_loader_p) -> float:
        """Baseline: FC-head angular error vs clean GT on pathological val set."""
        self.cnn.eval(); self.fc.eval(); errs = []
        with torch.no_grad():
            for imgs, gz_c, _ in val_loader_p:
                imgs = imgs.to(self.device, non_blocking=True)
                if self.use_amp:
                    with torch.amp.autocast('cuda'):
                        pred = self.fc(self.cnn(imgs)).cpu().float().numpy()
                else:
                    pred = self.fc(self.cnn(imgs)).cpu().numpy()
                pred /= (np.linalg.norm(pred, axis=1, keepdims=True) + 1e-8)
                errs.append(angular_error_np(pred, gz_c.numpy()))
        return float(np.concatenate(errs).mean())

    def _build_patho_gpm(self, train_loader_p, label=""):
        """
        FIX-3: Build pathological GPM on current feature space.
        Called once, on stable features (before joint FT shifts the CNN).
        """
        self.cnn.eval(); fp, gn = [], []
        with torch.no_grad():
            for imgs, _, gz_n in train_loader_p:
                imgs = imgs.to(self.device, non_blocking=True)
                if self.use_amp:
                    with torch.amp.autocast('cuda'):
                        f = self.cnn(imgs).cpu().float().numpy()
                else:
                    f = self.cnn(imgs).cpu().numpy()
                fp.append(f); gn.append(gz_n.numpy())
                if sum(len(x) for x in fp) >= N_SAMPLES: break
        fp = np.concatenate(fp)[:N_SAMPLES]
        gn = np.concatenate(gn)[:N_SAMPLES]
        k  = min(N_NEIGHBORS, len(fp) - 1)
        self.gpm_patho = RobustGPM(n_neighbors=k,
                                    sa_epochs=SA_EPOCHS,
                                    n_restarts=SA_RESTARTS)
        self.gpm_patho.fit_isomap(fp)
        self.gpm_patho.fit_sphere_alignment(self.gpm_patho.source_pgf, gn)
        sa_err = angular_error_np(
            self.gpm_patho.predict(self.gpm_patho.source_pgf), gn).mean()
        flag = '✓' if sa_err < 10 else ('⚠' if sa_err < 14 else '✗')
        print(f"  Patho GPM SA in-sample ({label}): {sa_err:.2f}° {flag}")

    # ══ EVALUATION ═════════════════════════════════════════════════════════════

    def evaluate_dual(self, val_loader_clean, val_loader_patho):
        print(f"\n{'='*55}\nDual Evaluation\n{'='*55}")
        self.cnn.eval(); results = {}

        # A: Healthy
        print("\n  [A] Healthy model ...")
        fc_l, gz_l, ft_l = [], [], []
        with torch.no_grad():
            for imgs, gz in val_loader_clean:
                imgs = imgs.to(self.device, non_blocking=True)
                if self.use_amp:
                    with torch.amp.autocast('cuda'):
                        f  = self.cnn(imgs).cpu().float().numpy()
                        pf = self.fc(torch.FloatTensor(f).to(self.device)).cpu().numpy()
                else:
                    f  = self.cnn(imgs).cpu().numpy()
                    pf = self.fc(torch.FloatTensor(f).to(self.device)).cpu().numpy()
                pf /= (np.linalg.norm(pf, axis=1, keepdims=True) + 1e-8)
                ft_l.append(f); gz_l.append(gz.numpy()); fc_l.append(pf)
        feats_c = np.concatenate(ft_l)
        gazes_c = np.concatenate(gz_l)
        fc_pred = np.concatenate(fc_l)
        fc_err  = angular_error_np(fc_pred, gazes_c)
        results.update(fc=fc_err, fc_pred=fc_pred, gt_clean=gazes_c)
        print(f"  FC  Layer: {fc_err.mean():.2f}° ± {fc_err.std():.2f}°  "
              f"med={np.median(fc_err):.2f}°")

        if self.gpm:
            pgf_c = self.gpm.project_all_to_3d(feats_c)
            gp    = self.gpm.predict(pgf_c)
            ge    = angular_error_np(gp, gazes_c)
            impr  = (fc_err.mean()-ge.mean())/fc_err.mean()*100
            results.update(gpm_healthy=ge, gpm_healthy_pred=gp)
            print(f"  GPM healthy: {ge.mean():.2f}° ± {ge.std():.2f}°  ({impr:+.1f}%)")

        # B: Pathological
        print("\n  [B] Pathological model ...")
        fp_l, gc_l, gn_l, mu_l, ka_l = [], [], [], [], []
        with torch.no_grad():
            for imgs, gz_c, gz_n in val_loader_patho:
                imgs = imgs.to(self.device, non_blocking=True)
                if self.use_amp:
                    with torch.amp.autocast('cuda'):
                        f         = self.cnn(imgs).cpu().float().numpy()
                        mu, kappa = self.vmf_head(torch.FloatTensor(f).to(self.device))
                else:
                    f         = self.cnn(imgs).cpu().numpy()
                    mu, kappa = self.vmf_head(torch.FloatTensor(f).to(self.device))
                fp_l.append(f); gc_l.append(gz_c.numpy())
                gn_l.append(gz_n.numpy())
                mu_l.append(mu.cpu().numpy())
                ka_l.append(kappa.cpu().float().numpy().flatten())
        feats_p   = np.concatenate(fp_l); gazes_p = np.concatenate(gc_l)
        gazes_n   = np.concatenate(gn_l); mu_all  = np.concatenate(mu_l)
        kappa_all = np.concatenate(ka_l)
        vmf_ec    = angular_error_np(mu_all, gazes_p)  # vs clean GT (key metric)
        vmf_en    = angular_error_np(mu_all, gazes_n)  # vs noisy GT
        mk        = float(kappa_all.mean())
        results.update(vmf_pred=mu_all, vmf_kappa=kappa_all,
                       vmf_err_vs_clean=vmf_ec, vmf_err_vs_noisy=vmf_en,
                       gt_noisy=gazes_n)
        print(f"  vMF μ (vs clean): {vmf_ec.mean():.2f}° ± {vmf_ec.std():.2f}°  "
              f"med={np.median(vmf_ec):.2f}°")
        print(f"  vMF μ (vs noisy): {vmf_en.mean():.2f}° ± {vmf_en.std():.2f}°")
        print(f"  κ mean/min/max  : {mk:.2f} / "
              f"{float(kappa_all.min()):.2f} / {float(kappa_all.max()):.2f}")
        print(f"  95%% cone       : {cone_95(mk):.1f}°")

        if self.gpm_patho:
            pgf_p = self.gpm_patho.project_all_to_3d(feats_p)
            gp_p  = self.gpm_patho.predict(pgf_p)
            ge_p  = angular_error_np(gp_p, gazes_n)
            results.update(gpm_patho=ge_p, gpm_patho_pred=gp_p)
            print(f"  GPM patho: {ge_p.mean():.2f}° ± {ge_p.std():.2f}°")

        # Full publishable table
        print(f"\n  {'─'*58}")
        print(f"  {'Method':<28} {'Mean':>8} {'Std':>8} {'Median':>8}")
        print(f"  {'─'*58}")
        for name, arr in [
            ("FC baseline (healthy)",     fc_err),
            ("GPM-AGG (healthy)",         results.get('gpm_healthy', np.zeros(1))),
            ("GPM-AGG (pathological)",    results.get('gpm_patho',   np.zeros(1))),
            ("vMF μ vs clean GT",         vmf_ec),
            ("vMF μ vs noisy GT",         vmf_en),
        ]:
            print(f"  {name:<28} {arr.mean():>7.2f}° "
                  f"{arr.std():>7.2f}° {np.median(arr):>7.2f}°")
        print(f"  {'─'*58}")
        print(f"  κ: {mk:.2f}  "
              f"(min={float(kappa_all.min()):.2f}  "
              f"max={float(kappa_all.max()):.2f})  "
              f"95°cone={cone_95(mk):.1f}°")
        print(f"  {'─'*58}")
        return results

    def export_results(self, results, path='gaze_results_dual.json', n_vis=200):
        out = dict(condition=self.condition, severity=self.severity,
                   kappa_clamp=[KAPPA_MIN, KAPPA_MAX],
                   sa_restarts=SA_RESTARTS, sa_epochs=SA_EPOCHS,
                   summary={}, samples_healthy=[], samples_pathological=[])
        s = out['summary']
        if 'fc' in results:
            fe=results['fc']
            s.update(fc_mean=round(float(fe.mean()),3), fc_std=round(float(fe.std()),3),
                     fc_median=round(float(np.median(fe)),3))
        if 'gpm_healthy' in results:
            ge=results['gpm_healthy']
            s.update(gpm_healthy_mean=round(float(ge.mean()),3),
                     gpm_healthy_std =round(float(ge.std()),3),
                     improvement_pct =round(float(
                         (results['fc'].mean()-ge.mean())/results['fc'].mean()*100),2))
        if 'gpm_patho' in results:
            gp=results['gpm_patho']
            s.update(gpm_patho_mean=round(float(gp.mean()),3),
                     gpm_patho_std =round(float(gp.std()),3))
        if 'vmf_kappa' in results:
            ka=results['vmf_kappa']
            s.update(vmf_kappa_mean   =round(float(ka.mean()),3),
                     vmf_kappa_min    =round(float(ka.min()),3),
                     vmf_kappa_max    =round(float(ka.max()),3),
                     vmf_cone_95_deg  =round(cone_95(float(ka.mean())),2),
                     vmf_err_vs_clean =round(float(results['vmf_err_vs_clean'].mean()),3),
                     vmf_err_vs_noisy =round(float(results['vmf_err_vs_noisy'].mean()),3))
        if all(k in results for k in ('gt_clean','gpm_healthy_pred','fc_pred','gpm_healthy','fc')):
            n = min(n_vis, len(results['gt_clean']))
            out['samples_healthy'] = [
                dict(id=int(i),
                     gt     =[round(float(v),4) for v in results['gt_clean'][i]],
                     gpm    =[round(float(v),4) for v in results['gpm_healthy_pred'][i]],
                     fc     =[round(float(v),4) for v in results['fc_pred'][i]],
                     gpm_err=round(float(results['gpm_healthy'][i]),2),
                     fc_err =round(float(results['fc'][i]),2))
                for i in range(n)]
        if all(k in results for k in ('gt_noisy','vmf_pred','vmf_kappa',
                                       'vmf_err_vs_noisy','vmf_err_vs_clean')):
            n  = min(n_vis, len(results['gt_noisy']))
            ka = results['vmf_kappa']
            out['samples_pathological'] = [
                dict(id=int(i),
                     gt_noisy    =[round(float(v),4) for v in results['gt_noisy'][i]],
                     vmf_mu      =[round(float(v),4) for v in results['vmf_pred'][i]],
                     kappa       =round(float(ka[i]),3),
                     cone_95_deg =round(cone_95(float(ka[i])),2),
                     err_vs_noisy=round(float(results['vmf_err_vs_noisy'][i]),2),
                     err_vs_clean=round(float(results['vmf_err_vs_clean'][i]),2))
                for i in range(n)]
        with open(path, 'w') as f: json.dump(out, f, indent=2)
        print(f"  Exported → {path}")
        return out

    def run_full_pipeline(self, train_loader, val_loader, train_ds, val_ds):
        print("\n" + "="*55
              + f"\nDUAL AGG | {self.condition} | {DEVICE}\n" + "="*55)
        t0    = time.time()
        eff_n = min(N_SAMPLES, len(train_ds))
        self.pretrain(train_loader, val_loader)
        self.build_gpm(train_loader, n_samples=eff_n, n_neighbors=N_NEIGHBORS)
        self.train_ip(n_samples=eff_n)
        self.sphere_oriented_training(train_loader)
        tl_p, vl_p = self.finetune_pathological(train_ds, val_ds)
        results = self.evaluate_dual(val_loader, vl_p)
        print(f"\n✓ Done: {(time.time()-t0)/60:.1f} min total")
        self.export_results(results, f'gaze_results_{self.condition}.json')
        self._save_ckpt('final')
        return results

# ═══════════════════════════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════════════════════════

def main():
    print("="*55)
    print(f"AGG + Neuropalsy FINAL | GPU={USE_CUDA} | batch={BATCH_SIZE}")
    print(f"SA: {SA_RESTARTS} restarts × {SA_EPOCHS} ep  | IP: {IP_EP} ep")
    print(f"κ∈[{KAPPA_MIN},{KAPPA_MAX}]  | overfit_gap={OVERFIT_GAP}")
    print("="*55)

    if not os.path.exists(DATASET_PATH):
        print(f"ERROR: {DATASET_PATH}"); return

    train_ds = MPIIFaceGazeDataset(DATASET_PATH, split='train', ratio=TRAIN_RATIO)
    val_ds   = MPIIFaceGazeDataset(DATASET_PATH, split='test',  ratio=TRAIN_RATIO)

    nw = NUM_WORKERS if os.name != 'nt' else 0
    def make_loader(ds, shuffle):
        kw = dict(batch_size=BATCH_SIZE, shuffle=shuffle,
                  num_workers=nw, pin_memory=USE_CUDA,
                  persistent_workers=(nw > 0))
        if USE_CUDA and nw > 0: kw['prefetch_factor'] = 2
        return DataLoader(ds, **kw)

    train_loader = make_loader(train_ds, shuffle=True)
    val_loader   = make_loader(val_ds,   shuffle=False)

    conditions = [('nystagmus', 0.6)]   # add more after first run confirms it works
    all_results = {}

    for condition, severity in conditions:
        print(f"\n{'#'*55}\n# {condition.upper()}  severity={severity}\n{'#'*55}")
        agg = DualAGGFramework(condition=condition, severity=severity)
        res = agg.run_full_pipeline(train_loader, val_loader, train_ds, val_ds)
        all_results[condition] = res

    print("\n" + "="*58 + "\nFINAL PUBLISHABLE TABLE\n" + "="*58)
    print(f"{'Condition':<14} {'FC°':>6} {'GPM_H°':>7} {'GPM_P°':>7} "
          f"{'vMF_c°':>7} {'vMF_n°':>7} {'κ':>6} {'95°':>6}")
    print("─"*58)
    for cond, res in all_results.items():
        fc    = res.get('fc',             np.zeros(1)).mean()
        gpm_h = res.get('gpm_healthy',    np.zeros(1)).mean()
        gpm_p = res.get('gpm_patho',      np.zeros(1)).mean()
        vmf_c = res.get('vmf_err_vs_clean', np.zeros(1)).mean()
        vmf_n = res.get('vmf_err_vs_noisy', np.zeros(1)).mean()
        kappa = res.get('vmf_kappa',      np.zeros(1)).mean()
        print(f"{cond:<14} {fc:>5.2f}° {gpm_h:>6.2f}° {gpm_p:>6.2f}° "
              f"{vmf_c:>6.2f}° {vmf_n:>6.2f}° {kappa:>6.2f} {cone_95(float(kappa)):>5.1f}°")
    return all_results

if __name__ == '__main__':
    results = main()

In [ ]:
import json

with open('/kaggle/working/gaze_results_nystagmus.json') as f:
    data = json.load(f)

In [ ]:
print(data['summary'])

In [ ]:
import pandas as pd

summary_df = pd.DataFrame([data['summary']])
summary_df.to_csv('/kaggle/working/summary.csv', index=False)

summary_df

In [ ]:
samples_patho = pd.DataFrame(data['samples_pathological'])
samples_healthy = pd.DataFrame(data['samples_healthy'])

samples_patho.to_csv('/kaggle/working/samples_pathological.csv', index=False)
samples_healthy.to_csv('/kaggle/working/samples_healthy.csv', index=False)

samples_patho.head()

In [ ]:
import numpy as np

def vector_to_angles(v):
    x, y, z = v
    yaw = np.arctan2(x, z)
    pitch = np.arcsin(y)
    return yaw, pitch

df['yaw'], df['pitch'] = zip(*df['vmf_mu'].apply(vector_to_angles))

df.head()